# Adversarial Sea-Surface-Height Reconstruction Experiment

Curated from the archived research notebook `new UCSD-1D-GAN.ipynb`. Read the repository
README and `docs/limitations.md` before execution. All original files and
execution outputs were preserved separately.

The original workflow monitors arrays named `testing` during training.
Its displayed validation curves are not an independent final test. Training
is disabled until `ALLOW_TRAINING` is explicitly enabled.


In [ ]:
from pathlib import Path
import os
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/project_paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or one of its notebook directories.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import NotebookPaths
paths = NotebookPaths(PROJECT_ROOT, output_group='models/adversarial_reconstruction')
input_path, input_glob, output_path = paths.input_path, paths.input_glob, paths.output_path
ALLOW_TRAINING = False  # Explicitly enable before running model training cells.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import scipy
from glob2 import glob


In [ ]:
file_path = 'new_ssh_training_data.nc'
ds = xr.open_dataset(input_path(file_path))
training = ds["ssh_training_data"].values
training_target =ds["ssh_training_target"].values
print(training.shape)


In [ ]:
file_path = 'new_ssh_testing_data.nc'
ds = xr.open_dataset(input_path(file_path))
testing = ds["ssh_testing_data"].values
testing_target =ds["ssh_testing_target"].values
print(testing.shape)


In [ ]:
import tensorflow as tf
import tensorflow.keras as tfkeras
from tensorflow.keras import regularizers
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
from matplotlib.pyplot import colorbar
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, BatchNormalization, MaxPooling2D, UpSampling2D, Reshape, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping

# Split the data into training and testing sets
training = np.transpose(training, (2, 1, 0))
training_target = np.transpose(training_target, (2, 1, 0))
testing = np.transpose(testing, (2, 1, 0))
testing_target = np.transpose(testing_target, (2, 1, 0))
train, validation, train_target, validation_target = train_test_split(training, training_target, test_size=0.1, random_state=42)

# Check data types and shapes
print(f'x_train shape: {train.shape}, dtype: {train.dtype}')
print(f'y_train shape: {train_target.shape}, dtype: {train_target.dtype}')
print(f'x_test shape: {validation.shape}, dtype: {validation.dtype}')
print(f'y_test shape: {validation_target.shape}, dtype: {validation_target.dtype}')


In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Flatten, Dense, LeakyReLU
from tensorflow.keras.optimizers import Adam, SGD

# Define the generator (for denoising)
def build_generator(input_shape):
    input_img = Input(shape=input_shape)

    # Encoder
    x = Conv2D(16, (4, 8), padding='same')(input_img)
    encoded = Conv2D(64, (4, 8), padding='same')(x)

    # Decoder
    x = Conv2DTranspose(64, (4, 8), padding='same')(encoded)
    x = Conv2DTranspose(16, (4, 8), padding='same')(x)
    decoded = Conv2D(1, (4, 8), padding='same')(x)

    return Model(input_img, decoded, name='generator')

# Define the discriminator
def build_discriminator(input_shape):
    input_img = Input(shape=input_shape)
    x = Conv2D(32, (4, 8), padding='same')(input_img)
    x = Conv2D(64, (4, 8), padding='same')(x)
    x = Flatten()(x)
    x = Dense(1)(x)

    return Model(input_img, x, name='discriminator')

# Input shape
input_shape = (train.shape[1], train.shape[2], 1)

# Instantiate the generator and discriminator
generator = build_generator(input_shape)
discriminator = build_discriminator(input_shape)

# Make the discriminator non-trainable when training the GAN model
discriminator.trainable = True

# Define the GAN model
gan_input = Input(shape=input_shape)
denoised_img = generator(gan_input)
gan_output = discriminator(denoised_img)

gan = Model(gan_input, gan_output, name='GAN')
# Compile the discriminator with SGD
discriminator.compile(optimizer=SGD(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# GAN model is compiled with Adam
gan.compile(optimizer=Adam(learning_rate=0.00002), loss='mean_absolute_error')

# Print the summaries of the models
print("Generator:")
generator.summary()

print("\nDiscriminator:")
discriminator.summary()

print("\nGAN:")
gan.summary()


In [ ]:
if not ALLOW_TRAINING:
    raise RuntimeError('Set ALLOW_TRAINING=True after reviewing the epoch count and validation protocol.')

import numpy as np
import matplotlib.pyplot as plt

# Hyperparameters
epochs = 50
batch_size = 16

# Labels for real and fake data
real_labels = np.ones((batch_size, 1))
fake_labels = np.zeros((batch_size, 1))

# Initialize lists to keep track of losses
d_losses = []
g_losses = []

# GAN Training Loop
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")

    for i in range(0, len(train), batch_size):
        # Ensure the batch size matches
        end = min(i + batch_size, len(train))
        batch_size_actual = end - i

        # Select a batch of real samples
        real_imgs = train_target[i:end]
        real_labels_batch = np.ones((batch_size_actual, 1))  # Match the actual batch size

        # Get corresponding noisy images
        noisy_imgs = train[i:end]

        # Generate a batch of fake (denoised) images
        fake_imgs = generator.predict(noisy_imgs)

        # Create fake labels for this batch
        fake_labels_batch = np.zeros((batch_size_actual, 1))  # Match the actual batch size

        # Train the discriminator
        d_loss_real = discriminator.train_on_batch(real_imgs, real_labels_batch)
        d_loss_fake = discriminator.train_on_batch(fake_imgs, fake_labels_batch)
        d_loss = 0.5 * np.add(d_loss_real[0], d_loss_fake[0])  # Average loss
        d_losses.append(d_loss)

        # Train the generator
        g_loss = gan.train_on_batch(noisy_imgs, real_labels_batch)
        g_losses.append(g_loss)

    # Print progress for each epoch
    print(f"Epoch {epoch + 1}/{epochs} | D loss: {d_loss:.4f} | G loss: {g_loss:.4f}")


In [ ]:
# Plot training history
plt.plot(d_losses, color='blue', label='Discriminator Loss')
plt.plot(g_losses, color='red', label='Generator Loss')
plt.legend()
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.ylim(0, 1)
plt.show()


In [ ]:
# Save the entire model
gan.save(output_path('gan.h5'))


In [ ]:
denoised_ssha = generator.predict(testing)
# Reshape the denoised images back to the original shape without the channel dimension
denoised_ssha = np.squeeze(denoised_ssha, axis=-1)
print(tf.keras.losses.MeanAbsoluteError()(testing , testing_target))
print(tf.keras.losses.MeanAbsoluteError()(denoised_ssha, testing_target))
